Aggregates solo, intact, and partial elements across all species into three master tables: data/solo_elements.tsv, data/intact_elements_dante.tsv, data/partial_elements_dante.tsv. 

Parses DANTE-LTR GFF3 attributes. Joins solo LTR class assignments via lib.LTR.info lookup.

Input: Per-species solo_list files, DANTE-LTR GFF3s, chromosome_sizes.tsv

In [1]:
import os
import glob
import pandas as pd

In [2]:
chr_names_path = "data/chromosome_sizes.tsv"
chr_names_df = pd.read_csv(chr_names_path, sep='\t')
included_accessions = chr_names_df['chr'].tolist()

# unique sample names from species comlumn
samples = chr_names_df['species'].unique().tolist()

# Solo LTRs

In [3]:
solo_columns = ['chr', 'start', 'end', 'id', 'library_element', 'coverage']
all_dfs = []  # Store DataFrames for efficient concatenation

for sample in samples:
    solo_files = glob.glob(f'data/solo_ratio/{sample}/*.solo_list')
    library_path = f'data/ltr_library/{sample}/lib.LTR.info'
    
    # Read and preprocess library file
    library = pd.read_csv(library_path, sep='\t', header=None, names=['library_element', 'start', 'end', 'length'])
    library['element_id'] = library['library_element'].str.split('#').str[0]
    library['Class'] = library['library_element'].str.split('#').str[1].str.split('|').str[2:].str.join('|')
    library = library[['element_id', 'Class']]  # Keep only necessary columns
    library = library.drop_duplicates(subset=['element_id'])
    
    sample_dfs = []  # Store DataFrames for this sample
    
    for file in solo_files:
        df = pd.read_csv(file, sep='\t', header=None, names=solo_columns)
        df['sample'] = sample
        
        # Merge instead of iterating
        df = df.merge(library, left_on='library_element', right_on='element_id', how='left')
        
        sample_dfs.append(df)
    
    # Concatenate once per sample
    if sample_dfs:
        all_dfs.append(pd.concat(sample_dfs, ignore_index=True))

# Final concatenation
solo_elements = pd.concat(all_dfs, ignore_index=True)

# filter included accessions
solo_elements = solo_elements[solo_elements['chr'].isin(included_accessions)]

solo_elements.to_csv('data/solo_elements.tsv', index=False, sep='\t')

In [4]:
#create a list of chromosomes where solo-LTR elements are found
accessions = solo_elements['chr'].unique().tolist()

In [5]:
intact_elements = pd.DataFrame()
partial_elements = pd.DataFrame()
columns = ['chr','source','feature_type','start','end','score','strand','phase','attributes']


def extract_intact(df, accessions):
    # filter by accessions
    df_filt = df.loc[df["chr"].isin(accessions)].copy()

    TEs = df_filt.loc[df_filt["feature_type"] == "transposable_element", ["sample","chr","start","end","attributes"]].copy()
    LTRs = df_filt.loc[df_filt["feature_type"] == "long_terminal_repeat", ["sample","attributes"]].copy()


    # parse attributes (Series-returning extract)
    TEs["ID"]    = TEs["attributes"].str.extract(r"ID=([^;]+)", expand=False)
    TEs["Class"] = TEs["attributes"].str.extract(r"Final_Classification=([^;]+)", expand=False) \
                                   .str.split("|").str[2:].str.join("|")
    TEs["Rank"]  = TEs["attributes"].str.extract(r"Rank=([^;]+)", expand=False)
    TEs["LTR_length"] = pd.to_numeric(
        TEs["attributes"].str.extract(r"LTR5_length=(\d+)", expand=False), errors="coerce"
    )   

    LTRs["LTR_identity"] = pd.to_numeric(
        LTRs["attributes"].str.extract(r"LTR_Identity=([\d.]+)", expand=False), errors="coerce"
    )
    LTRs["Parent"] = LTRs["attributes"].str.extract(r"Parent=([^;]+)", expand=False)

    LTRs = LTRs.dropna(subset=["Parent","LTR_identity"]).drop_duplicates(subset=["sample","Parent"])

    # join on ID/Parent + sample
    TEs = TEs.merge(LTRs[["sample","Parent","LTR_identity"]],
                    left_on=["sample","ID"], right_on=["sample","Parent"], how="left",
                    validate="m:1").drop(columns=["Parent"])

    mask_partial = TEs["ID"].str.contains("partial", na=False)
    cols = ["ID","sample","chr","start","end","Class","Rank","LTR_identity","LTR_length"]
    D_rank  = TEs.loc[mask_partial, cols].copy()
    DL_rank = TEs.loc[~mask_partial, cols].copy()
    return D_rank, DL_rank


for sample in samples:
    intact_files = glob.glob(f'data/ltr_samplewise/{sample}.tes.gff3')
    df = pd.read_csv(intact_files[0], sep='\t', header=None, names=columns, comment='#')
    df['sample'] = sample
    D_rank_df, DL_rank_df = extract_intact(df, accessions)
    intact_elements = pd.concat([intact_elements, DL_rank_df])
    partial_elements = pd.concat([partial_elements, D_rank_df])

# filter included accessions
intact_elements = intact_elements[intact_elements['chr'].isin(included_accessions)]
partial_elements = partial_elements[partial_elements['chr'].isin(included_accessions)]
# save outputs
intact_elements.to_csv("data/intact_elements_dante.tsv", index=False, sep='\t')
partial_elements.to_csv("data/partial_elements_dante.tsv", index=False, sep='\t')